# Track B: Agentic AI MLOps — Demo Notebook

This notebook demonstrates the complete MLOps pipeline:
1. AgentTracer setup (MLflow)
2. Prompt versioning (v1 → v2 → v3)
3. Real assistant execution with tool calling
4. Metrics comparison
5. Evidently regression testing

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

## 1. Initialize AgentTracer

In [ ]:
from src.track_b.utils.agent_tracer import AgentTracer

tracking_uri = f'sqlite:///{project_root / "data" / "mlflow.db"}'
tracer = AgentTracer(tracking_uri=tracking_uri)
experiment_id = tracer.get_or_create_experiment('AgenticMLOps')
print(f'Experiment ID: {experiment_id}')
print(f'Tracking URI: {tracking_uri}')

## 2. Load Prompt Versions

Each version addresses a specific failure mode found in testing.

In [ ]:
prompts_dir = project_root / 'src' / 'track_b' / 'prompts'

prompt_v1 = (prompts_dir / 'prompt_v1.txt').read_text()
prompt_v2 = (prompts_dir / 'prompt_v2.txt').read_text()
prompt_v3 = (prompts_dir / 'prompt_v3.txt').read_text()

print(f'V1 ({len(prompt_v1)} chars): Baseline prompt')
print(f'V2 ({len(prompt_v2)} chars): + Always call tool rule')
print(f'V3 ({len(prompt_v3)} chars): + Chain searches + Verify claims')
print(f'\nV2 adds {len(prompt_v2) - len(prompt_v1)} chars over V1')
print(f'V3 adds {len(prompt_v3) - len(prompt_v1)} chars over V1')

## 3. Run Experiment with Real Assistant

In [ ]:
import asyncio
import time
from src.track_b.assistant.config import settings
from src.track_b.assistant.llm.provider import get_provider
from src.track_b.assistant.agent.loop import AgenticLoop
from src.track_b.assistant.tools.registry import tool_registry
from src.track_b.assistant.tools.calculator import calculator_tool
from src.track_b.assistant.tools.web_search import web_search_tool, datetime_tool

# Register tools
for tool in [calculator_tool, web_search_tool, datetime_tool]:
    if tool['name'] not in tool_registry.list_tools():
        tool_registry.register(tool['name'], tool['description'], tool['parameters'], tool['executor'])

provider = get_provider(settings.llm_provider)
test_queries = [
    'What is Retrieval-Augmented Generation (RAG)?',
    'What is the current date and time?',
    'Calculate 15% of 340.',
]
print(f'Provider: {settings.llm_provider}, Model: {settings.groq_model}')
print(f'Test queries: {len(test_queries)}')

In [ ]:
async def run_version(version_name, prompt_template):
    results = []
    for query in test_queries:
        loop = AgenticLoop(provider, max_iterations=3, system_prompt=prompt_template)
        result = await loop.run(query)
        results.append({
            'query': query,
            'response': result.answer[:200],
            'iterations': result.total_iterations,
            'stopped': result.stopped_reason,
            'tools': [s.tool_name for s in result.steps if s.action == 'tool_call'],
        })
        await asyncio.sleep(3)
    completed = sum(1 for r in results if r['stopped'] == 'model_answered')
    return {'version': version_name, 'results': results, 'success_rate': completed / len(results)}

# Run all versions
versions = {'v1': prompt_v1, 'v2': prompt_v2, 'v3': prompt_v3}
all_results = {}
for name, template in versions.items():
    print(f'\nRunning {name}...')
    r = asyncio.run(run_version(name, template))
    all_results[name] = r
    print(f'  Success rate: {r["success_rate"]:.0%}')

## 4. Log to MLflow

In [ ]:
import mlflow

for name, result in all_results.items():
    run = tracer.get_run(f'prompt_{name}', experiment_id)
    tracer.log_run(
        run,
        params={'prompt_version': name, 'prompt_length': len(versions[name])},
        metrics={'task_success_rate': result['success_rate'], 'num_queries': len(result['results'])},
        tags={'prompt_version': name}
    )
    tracer.log_trace(run, {'version': name, 'results': result['results']}, f'trace_{name}')
    tracer.end_run(run.info.run_id, 'FINISHED')
    print(f'Logged {name}: run_id={run.info.run_id}')

## 5. Compare Results

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {'Version': name, 'Success Rate': r['success_rate'], 'Avg Iterations': sum(x['iterations'] for x in r['results']) / len(r['results'])}
    for name, r in all_results.items()
])
print('\n=== Version Comparison ===')
print(comparison.to_string(index=False))
print('\nMLflow UI: http://localhost:5000')

## 6. Evidently Regression Testing

In [ ]:
from src.track_b.utils.evidently_judge import EvidentlyJudge, build_dataset
import pandas as pd

golden = [
    {'query': 'What is RAG?', 'target': 'RAG combines retrieval from an external knowledge base with LLM generation.'},
    {'query': 'What is the current date?', 'target': 'The current date can be determined using the get_current_datetime tool.'},
    {'query': 'Calculate 15% of 340.', 'target': '15% of 340 is 51.'},
]

for name, result in all_results.items():
    judge = EvidentlyJudge(reports_dir=project_root / 'reports')
    eval_df = pd.DataFrame([
        {'query': r['query'], 'new_response': r['response'], 'target_response': g['target']}
        for r, g in zip(result['results'], golden)
    ])
    quality = judge.evaluate_judge_quality(eval_df, 'new_response', 'target_response', 'query')
    print(f'{name}: Judge agreement = {quality["agreement_rate"]:.1%}')

## Summary

- **v1 (Baseline)**: Fastest, lowest token usage
- **v2 (+ Tool enforcement)**: Adds reliability for factual queries
- **v3 (+ Chain searches)**: Best for complex multi-source questions

All versions logged to MLflow with traces, metrics, and Evidently reports.